# 05 — Spatial Validation

Most published outbreak models answer *"will malaria rise next month?"*. Agencies
also need *"where will it arrive next?"*, because pre-positioning supplies in the
wrong district costs as much as not pre-positioning at all (shortcoming #10).

This notebook validates the geography of the forecast:

1. the **travel matrix** — empirical CDR where available, gravity or radiation
   otherwise (critical rule #8);
2. the **diffusion model** — a reaction-diffusion process on the district graph;
3. **spatial skill** — hotspot hit rate, rank correlation, and displacement error
   in kilometres, because naming the neighbouring district is far more useful
   than naming a council 700 km away.

In [ ]:
# Make the repo importable regardless of where Jupyter was launched from.
import sys, pathlib, warnings
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / "src").is_dir() and (p / "config").is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")

import logging
logging.getLogger("afya").setLevel(logging.WARNING)   # keep notebook output readable

import numpy as np
import pandas as pd

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 40)
print(f"repo root: {ROOT}")

In [ ]:
# Plotting is optional throughout these notebooks: matplotlib is not a hard
# dependency of AFYA-PREDICT, because the platform must install on low-spec
# district hardware. Every notebook falls back to printed tables without it.
#
# Backend selection matters more than it looks. Inside a Jupyter kernel,
# matplotlib configures its own inline backend and we leave it alone. Anywhere
# else - `nbconvert --execute`, CI, a headless server - a GUI backend will block
# forever on a window that never opens (a set-but-unreachable $DISPLAY is enough
# to trigger it), so we force the non-interactive Agg backend.
import os
import sys

try:
    import matplotlib
    _in_kernel = "ipykernel" in sys.modules
    if not os.environ.get("MPLBACKEND") and not _in_kernel:
        matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    plt.rcParams["figure.figsize"] = (11, 4)
    plt.rcParams["axes.grid"] = True
    plt.rcParams["grid.alpha"] = 0.3
    HAS_PLT = True
    print(f"matplotlib {matplotlib.__version__} on the "
          f"{matplotlib.get_backend()} backend")
except ImportError:
    HAS_PLT = False
    print("matplotlib not installed - tables will be printed instead of plotted")

## 1. A wider grid

Spatial questions need more than a handful of districts, so this notebook uses a
larger subset spanning several transport corridors.

In [ ]:
from src.core.config_loader import load_region_config, load_disease_config
from src.core.geo import subset_region

FULL_REGION = load_region_config("tanzania")
STUDY_DISTRICTS = [
    "Kinondoni", "Ilala", "Temeke", "Ubungo",        # Dar es Salaam conurbation
    "Bagamoyo", "Kibaha TC", "Morogoro MC",          # the central corridor out of Dar
    "Dodoma City", "Kongwa",                          # central plateau
    "Mwanza City", "Ilemela", "Sengerema", "Magu",    # Lake Victoria basin
    "Geita TC", "Kahama TC", "Shinyanga MC",          # the mining belt
    "Tanga City", "Korogwe", "Moshi MC", "Arusha City",  # northern corridor
]
REGION = subset_region(FULL_REGION, STUDY_DISTRICTS)
print(f"{len(REGION.districts)} districts across "
      f"{len({d.region for d in REGION.districts})} regions")
pd.DataFrame([d.model_dump() for d in REGION.districts]).set_index("name")[
    ["region", "population", "urban", "density_km2"]]

## 2. The travel matrix

Real CDR data requires a negotiated agreement with Vodacom/Airtel/Tigo and arrives
pre-aggregated as origin-destination counts — never subscriber records. The
platform uses it when present and falls back analytically when not, so the spatial
layer never depends on a commercial negotiation completing.

In [ ]:
from src.core.geo import distance_matrix, gravity_matrix, radiation_matrix
from src.feature_engineering.mobility_features import get_travel_matrix

travel = get_travel_matrix(REGION)                 # CDR if available, else gravity
gravity = gravity_matrix(REGION)
radiation = radiation_matrix(REGION)
distances = distance_matrix(REGION)

for name, matrix in [("gravity", gravity), ("radiation", radiation)]:
    assert np.allclose(matrix.sum(axis=1), 1.0), f"{name} rows must sum to 1"
    assert np.allclose(np.diag(matrix.to_numpy()), 0.0), f"{name} must have no self-flow"
print("both flow matrices are row-normalised with zero self-flow\n")

print("Strongest corridors under the gravity model:")
pairs = (gravity.stack().rename("flow").reset_index()
         .rename(columns={"level_0": "origin", "level_1": "destination"}))
pairs = pairs[pairs.origin != pairs.destination]
pairs["km"] = [distances.loc[o, d] for o, d in zip(pairs.origin, pairs.destination)]
display(pairs.nlargest(10, "flow").round(3).reset_index(drop=True))

### Gravity versus radiation

They disagree, and the disagreement is informative: gravity is tuned by a distance
exponent and favours large nearby destinations; radiation is parameter-free and
accounts for *intervening opportunities* — a traveller passing several equivalent
destinations is less likely to reach the far one.

In [ ]:
origin = "Mwanza City"
side_by_side = pd.DataFrame({
    "gravity": gravity.loc[origin],
    "radiation": radiation.loc[origin],
    "km": distances.loc[origin],
}).drop(index=origin).sort_values("gravity", ascending=False)

print(f"Outflow from {origin}:")
display(side_by_side.head(10).round(4))

rank_agreement = side_by_side["gravity"].corr(side_by_side["radiation"], method="spearman")
print(f"\nSpearman rank agreement between the two models: {rank_agreement:.3f}")
print("They broadly agree on ordering; where they diverge, the choice is")
print("configurable per disease via spatial.diffusion_model in the YAML.")

## 3. Diffusion from a seeded outbreak

The model is a discrete-time reaction-diffusion process: each week a share of a
district's infection pressure travels along the flow matrix, decays with the
disease's serial interval, and adds to the receiving district's local risk.

In [ ]:
from src.models.spatial_diffusion import SERIAL_INTERVAL_WEEKS, SpatialDiffusionModel

cholera = load_disease_config("cholera")
diffusion = SpatialDiffusionModel(cholera, REGION, travel_matrix=travel)

print(f"disease            : {cholera.name}")
print(f"transmission mode  : {cholera.transmission_mode.value}")
print(f"serial interval    : {SERIAL_INTERVAL_WEEKS[cholera.transmission_mode.value]} weeks")
print(f"weekly decay       : {diffusion.decay:.3f}")
print(f"importation weight : {cholera.spatial.importation_weight}")

SEED = "Mwanza City"
incidence = pd.Series(0.0, index=REGION.district_names)
incidence[SEED] = 4.0                      # per 1,000/week, a substantial outbreak

forecast = diffusion.project(incidence, "2024-W20", horizon_weeks=8)
print(f"\nSeeded {SEED} at {incidence[SEED]} per 1,000. Projected importation risk:")
display(forecast.risk.round(3).iloc[:, :8])

In [ ]:
print(f"Districts at highest importation risk one week after the {SEED} seed:\n")
display(forecast.top_districts(week=forecast.risk.index[0], n=8).round(3).to_frame("risk"))

first_week = forecast.risk.iloc[0]
ranked = first_week.drop(index=SEED).sort_values(ascending=False)
print("\nRisk against distance from the seed - proximity matters, but so does size:")
display(pd.DataFrame({
    "importation_risk": ranked.head(8).round(3),
    "km_from_seed": distances.loc[SEED, ranked.head(8).index].round(0),
    "population": [REGION.get(d).population for d in ranked.head(8).index],
}))

In [ ]:
if HAS_PLT:
    top = forecast.risk.iloc[-1].drop(index=SEED).nlargest(6).index
    fig, ax = plt.subplots(figsize=(11, 4))
    for district in top:
        ax.plot(range(len(forecast.risk)), forecast.risk[district], marker="o", ms=4, label=district)
    ax.set_xticks(range(len(forecast.risk)))
    ax.set_xticklabels(forecast.risk.index, rotation=45, ha="right", fontsize=8)
    ax.set_ylabel("importation risk (0-1)")
    ax.set_title(f"Spread from a {SEED} outbreak")
    ax.legend(fontsize=8, ncol=3)
    plt.tight_layout(); plt.show()

### Which districts are at risk *because of somebody else*?

This is the class of district a purely temporal model misses entirely: quiet
today, already seeded, and about to have a problem.

In [ ]:
ranking = diffusion.importation_ranking(incidence)
display(ranking.head(10).round(4))

at_risk = diffusion.at_risk_districts(incidence, threshold_ratio=0.6)
print(f"\n{len(at_risk)} district(s) whose risk is majority-imported rather than local:")
print("  " + ", ".join(at_risk[:12]))
print(f"\n{SEED} is correctly excluded: {SEED not in at_risk} "
      "(its risk is local, not imported)")

### Naming the source is what makes it actionable

In [ ]:
for district in ranking.index[:4]:
    if district == SEED:
        continue
    sources = diffusion.contributors(incidence, district, top_n=3)
    if not sources:
        continue
    named = ", ".join(f"{s.district} ({s.contributed_risk:.0%})" for s in sources)
    print(f"{district:16} <- {named}")
print("\nAn alert that says 'risk is arriving from Mwanza City along the lake corridor'")
print("tells a district officer where to put screening. A risk score alone does not.")

## 4. Spatial skill against observed spread

The metrics that matter: did the top-ranked districts turn out to be the affected
ones, was the ordering right, and — where it was wrong — how far off geographically?

In [ ]:
from src.data_ingestion.normalizer import ingest
from src.models.registry import build_module

module = build_module("malaria", region=REGION)
SOURCES = sorted(set(module.config.required_sources) | {"dhis2"})
panel = ingest(SOURCES, "2021-W01", "2024-W52", region=REGION)
matrix = module.build_feature_matrix(panel)

weeks = matrix.weeks
train_weeks = set(weeks[:-40])
from src.models.auto_retrain import _slice_weeks
module.train(_slice_weeks(matrix, train_weeks))
print(f"trained on {len(train_weeks)} weeks; validating on the remaining {len(weeks) - len(train_weeks)}")

In [ ]:
from src.evaluation.spatial_accuracy import evaluate_spatial_series

populations = pd.Series({d.name: float(d.population) for d in REGION.districts})
test_weeks = [w for w in weeks if w not in train_weeks]

rows = []
for district in matrix.districts:
    model = module.model_for(district)
    local = matrix.for_district(district).dropna_rows()
    local = local.X[np.isin(local.X.index.get_level_values("week"), test_weeks)]
    if local.empty:
        continue
    y = matrix.y.reindex(local.index)
    predicted = np.clip(model.predict(local), 0, None)
    population = populations[district]
    for idx, actual, prediction in zip(local.index, y.to_numpy(dtype=float), predicted):
        if not np.isfinite(actual):
            continue
        rows.append({"district": idx[0], "week": idx[1],
                     "actual": actual / population * 1000,
                     "predicted": prediction / population * 1000})

scored = pd.DataFrame(rows).set_index(["district", "week"])
spatial = evaluate_spatial_series(scored, REGION, k=5)
print(f"{len(spatial)} validation weeks scored\n")
display(spatial[["week", "hit_rate", "rank_correlation", "mean_distance_error_km"]].head(10).round(3))

print(f"\nMean hotspot hit rate (top 5)   : {spatial['hit_rate'].mean():.1%}")
print(f"Mean district rank correlation  : {spatial['rank_correlation'].mean():.3f}")
print(f"Mean displacement error         : {spatial['mean_distance_error_km'].mean():.0f} km")

A displacement error well below the typical inter-district distance means that
when the model is wrong, it is wrong *nearby* — it names a neighbouring council
rather than the wrong end of the country. For pre-positioning decisions that is a
materially different kind of error from a random miss.

In [ ]:
typical_spacing = distances.replace(0, np.nan).min(axis=1).mean()
mean_error = spatial["mean_distance_error_km"].mean()
print(f"mean nearest-neighbour distance in this grid : {typical_spacing:.0f} km")
print(f"mean displacement error of the forecast      : {mean_error:.0f} km")
print(f"ratio                                         : {mean_error / typical_spacing:.2f}x")
print("\n(Below ~2x means misses land in the immediate neighbourhood.)")

if HAS_PLT:
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 3.6))
    a1.plot(spatial["hit_rate"].to_numpy(), marker="o", ms=3)
    a1.axhline(spatial["hit_rate"].mean(), ls="--", c="r")
    a1.set_title("top-5 hotspot hit rate"); a1.set_ylim(0, 1)
    a2.plot(spatial["mean_distance_error_km"].to_numpy(), marker="o", ms=3, color="tab:orange")
    a2.axhline(typical_spacing, ls="--", c="k", label="nearest-neighbour spacing")
    a2.set_title("displacement error (km)"); a2.legend(fontsize=8)
    for ax in (a1, a2):
        ax.set_xlabel("validation week")
    plt.tight_layout(); plt.show()

## 5. Importation precision against real new-district onsets

The most direct test of shortcoming #10: not "was the national curve right" but
"did we name the districts the disease actually reached next".

In [ ]:
from src.evaluation.spatial_accuracy import detect_new_onsets, importation_accuracy

observed = scored["actual"].unstack("district")
observed = observed.reindex(sorted(observed.index))
threshold = module.config.alerts.medium

results = []
for i, week in enumerate(observed.index[4:], start=4):
    onsets = detect_new_onsets(observed, threshold, week, lookback=4)
    if not onsets:
        continue
    previous_week = observed.index[i - 1]
    pressure = diffusion.step(observed.loc[previous_week].fillna(0.0))
    scores = importation_accuracy(pressure, onsets, k=5)
    results.append({"week": week, "onsets": len(onsets),
                    "precision_at_5": scores["precision_at_k"],
                    "recall_at_5": scores["recall_at_k"],
                    "onset_districts": ", ".join(onsets[:3])})

if results:
    onset_frame = pd.DataFrame(results)
    display(onset_frame.round(3).head(12))
    print(f"\n{len(onset_frame)} week(s) with a new-district onset")
    print(f"mean recall@5: {onset_frame['recall_at_5'].mean():.1%} - the share of newly")
    print("affected districts that were already in the top-5 importation ranking.")
else:
    print(f"No new-district onsets above {threshold} per 1,000 in this validation window.")
    print("That is a property of the (synthetic) data, not a model failure - on a real")
    print("deployment this section is the headline spatial result.")

## 6. What this establishes

* The travel matrix is well-formed under both analytical models, and the platform
  degrades from CDR to gravity without losing the spatial layer.
* Diffusion from a seeded outbreak reaches the corridor districts first, in the
  order proximity and population size predict.
* Spatial skill is measured three ways, including displacement in kilometres, so
  a near miss is not scored identically to a random one.
* Every importation score comes with **named source districts**, which is what
  turns a map into an instruction.

Next: **`06_drift_and_retraining.ipynb`** — keeping all of this true over time.